# Real World RAG System — Legal Domain (`cuad`)

Improved pipeline (v2) over the team-share notebook. Upgrades:

1. **Hybrid retrieval** — BM25 + dense weighted fusion, α = 0.3 (arXiv:2407.01219)
2. **Stronger embedder** — `BAAI/bge-base-en-v1.5` with the proper bge query prefix
3. **Reverse repacking** — most relevant chunk closest to the question
4. **Chunk deduplication** — cuad reuses contracts across many rows
5. **Legal-tuned generation prompt** — targets answer completeness
6. **Wider retrieve → tighter rerank** (top_k=8 → rerank_k=4), clause-friendly splitting

Output CSV/JSON schema is identical to v1 so results are directly comparable.

**Required packages**

In [1]:
!pip install -q langchain langchain-community langchain-groq langchain-text-splitters \
    chromadb faiss-cpu flashrank sentence-transformers tqdm nltk pydantic \
    rank_bm25 langchain-openai datasets scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.1/122.1 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/13

**Imports**

In [2]:
import os
import json
import re
import time
import shutil
import hashlib
import numpy as np
import pandas as pd
from typing import List, Dict, Any

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from flashrank import Ranker, RerankRequest
from datasets import load_dataset
from sklearn.metrics import (
    mean_squared_error,
    root_mean_squared_error,
    accuracy_score,
    f1_score,
)

**Config — legal domain**

In [3]:
config = {
    "data_set": "cuad",                     # RAGBench legal subset
    "data_type": "test",
    "chunk_size": 1000,                     # chars (~250 tokens) — legal clauses are long
    "chunk_overlap": 200,
    "embedding_model": "BAAI/bge-base-en-v1.5",
    "retriever_type": "hybrid",             # "dense" | "sparse" | "hybrid"
    "hybrid_alpha": 0.3,                    # weight on sparse score (paper: 0.3 best)
    "top_k": 12,                             # retrieve wider, let reranker filter
    "ranking_model": "ms-marco-MiniLM-L-12-v2",
    "rerank_k": 4,
    "repacking_strategy": "reverse",        # paper: reverse > sides
    "domain": "legal",
    "generator_model": "llama-3.1-8b-instant",
    "judge_model": "llama-3.3-70b-versatile",
    "genai_provider": "groq",
}

config["experiment_tag"] = "v3_softprompt"   # per-document retrieval + judge key fix + prompt fix

FILE_FORMAT = (
    f"{config['data_set']}_{config['data_type']}_{config['chunk_size']}_"
    f"{config['chunk_overlap']}_{config['retriever_type']}_"
    f"{config['top_k']}_{config['repacking_strategy']}_{config['experiment_tag']}"
)
CHECKPOINT_FILE = f"ragbench_{FILE_FORMAT}_checkpoint.csv"
FINAL_FILE = f"ragbench_{FILE_FORMAT}_final.csv"
SUMMARY_FILE = f"ragbench_{FILE_FORMAT}_results_summary.json"

**Mount Drive & secrets** — paste your team Groq keys into `secret_keys`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/RAG_Project"
os.makedirs(DRIVE_DIR, exist_ok=True)
CHECKPOINT_DRIVE_FILE = f"{DRIVE_DIR}/{CHECKPOINT_FILE}"
FINAL_DRIVE_FILE = f"{DRIVE_DIR}/{FINAL_FILE}"
SUMMARY_DRIVE_FILE = f"{DRIVE_DIR}/{SUMMARY_FILE}"

secret_keys = [
]
secret_keys_index = 0
if secret_keys:
    os.environ["GROQ_API_KEY"] = secret_keys[secret_keys_index]

Mounted at /content/drive


**Util functions** — includes the new chunk deduplication (cuad reuses the same contracts across many rows).

In [5]:
def convert_dataframe_to_langchain_docs(df: pd.DataFrame) -> List[Document]:
    """RAGBench rows -> LangChain Documents, deduplicated.

    v1 indexed every row's documents even when identical passages
    appeared in many rows (common in cuad: one contract feeds many
    questions). Duplicates waste embedding time and crowd the top-k
    with copies. We dedup on a content hash but remember every row id
    the passage belongs to.
    """
    seen: Dict[str, Document] = {}
    for _, row in df.iterrows():
        row_id = str(row.get("id", "N/A"))
        dataset_name = str(row.get("dataset_name", "unknown"))
        passages = row.get("documents", [])
        joined = "\n\n".join(passages)

        h = hashlib.md5(joined.encode("utf-8")).hexdigest()
        if h in seen:
            seen[h].metadata["row_ids"] = (
                seen[h].metadata["row_ids"] + "," + row_id
            )
        else:
            seen[h] = Document(
                page_content=joined,
                metadata={
                    "id": row_id,
                    "row_ids": row_id,
                    "dataset_name": dataset_name,
                },
            )

    docs = list(seen.values())
    print(f"Converted {len(df)} rows -> {len(docs)} unique documents "
          f"({len(df) - len(docs)} duplicates removed).")
    return docs


def simple_sentence_split(text: str) -> List[str]:
    text = text.replace("\n", " ").strip()
    sentences = re.split(r"(?<=[.!?])\s+", text)
    return [s.strip() for s in sentences if len(s.strip()) > 0]


def key_context_sentences(retrieved_docs: List[Document]):
    """Assign RAGBench-style keys (0a, 0b, 1a, ...) to context sentences."""
    keyed_sentences, context_text = [], ""
    letters = "abcdefghijklmnopqrstuvwxyz"
    for doc_idx, doc in enumerate(retrieved_docs):
        sentences = simple_sentence_split(doc.page_content)
        context_text += f"\nDocument {doc_idx}:\n"
        for sent_idx, sent in enumerate(sentences):
            key = (f"{doc_idx}{letters[sent_idx]}"
                   if sent_idx < len(letters) else f"{doc_idx}{sent_idx}")
            keyed_sentences.append({"key": key, "sentence": sent})
            context_text += f"{key}: {sent}\n"
    return keyed_sentences, context_text


def key_response_sentences(response_text: str):
    sentences = simple_sentence_split(response_text)
    letters = "abcdefghijklmnopqrstuvwxyz"
    keyed_response, response_text_keyed = [], ""
    for i, sent in enumerate(sentences):
        key = letters[i] if i < len(letters) else str(i)
        keyed_response.append({"key": key, "sentence": sent})
        response_text_keyed += f"{key}: {sent}\n"
    return keyed_response, response_text_keyed


def parse_json_safely(text):
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None
    return None


def is_rate_limit_error(error) -> bool:
    error_text = str(error).lower()
    return any(p in error_text for p in
               ["rate limit", "rate_limit_exceeded", "tokens per day",
                "tpd", "429"])

**TRACe metrics computation** (identical to v1 for comparability)

In [6]:
def compute_trace_metrics(judge_output, keyed_context):
    # Only count keys that actually exist in the context — the judge
    # sometimes invents keys, which pushed relevance above 1.0
    valid_keys = {s["key"] for s in keyed_context}
    total_context_sentence_count = len(valid_keys)
    relevant_keys = set(judge_output.get("all_relevant_sentence_keys", [])) & valid_keys
    utilized_keys = set(judge_output.get("all_utilized_sentence_keys", [])) & valid_keys

    if total_context_sentence_count == 0:
        relevance_score = 0.0
        utilization_score = 0.0
    else:
        relevance_score = len(relevant_keys) / total_context_sentence_count
        utilization_score = len(utilized_keys) / total_context_sentence_count

    if len(relevant_keys) == 0:
        completeness_score = None
    else:
        completeness_score = (
            len(relevant_keys.intersection(utilized_keys)) / len(relevant_keys)
        )

    adherence_score = bool(judge_output.get("overall_supported", False))
    support_info = judge_output.get("sentence_support_information", [])
    if support_info:
        supported = sum(1 for s in support_info if s.get("fully_supported", False))
        adherence_continuous = supported / len(support_info)
    else:
        adherence_continuous = 1 if adherence_score else 0

    return {
        "pred_relevance_score": relevance_score,
        "pred_utilization_score": utilization_score,
        "pred_completeness_score": completeness_score,
        "pred_adherence_score": adherence_score,
        "pred_adherence_continuous": adherence_continuous,
    }

**Retrievers — dense, sparse, and NEW hybrid fusion**

`S = α · S_sparse + (1-α) · S_dense` with per-query min-max normalization.
α=0 → pure dense, α=1 → pure sparse, α=0.3 → hybrid (paper's best).
One class covers all three ablations — just change `retriever_type` in config.

In [7]:
class HuggingFaceEmbedder:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> np.ndarray:
        # normalize so dot product == cosine similarity; batch for speed
        return self.model.encode(
            texts, convert_to_numpy=True,
            normalize_embeddings=True,
            batch_size=64, show_progress_bar=True,
        )

    def embed_query(self, text: str) -> np.ndarray:
        # bge models retrieve better with this query instruction prefix
        prefixed = "Represent this sentence for searching relevant passages: " + text
        return self.model.encode(
            [prefixed], convert_to_numpy=True, normalize_embeddings=True
        )[0]


class SparseIndex:
    """BM25 over chunk tokens. Returns raw scores for ALL chunks."""

    def __init__(self, documents: List[Document]):
        self.documents = documents
        tokenized = [self._tokenize(d.page_content) for d in documents]
        self.bm25 = BM25Okapi(tokenized)

    @staticmethod
    def _tokenize(text: str) -> List[str]:
        return re.findall(r"[a-z0-9]+", text.lower())

    def scores(self, query: str) -> np.ndarray:
        return np.asarray(self.bm25.get_scores(self._tokenize(query)),
                          dtype=np.float64)


class DenseIndex:
    """Embedding index. Returns cosine scores for ALL chunks."""

    def __init__(self, documents: List[Document], embedder: HuggingFaceEmbedder):
        self.documents = documents
        self.embedder = embedder
        texts = [d.page_content for d in documents]
        self.doc_embeddings = embedder.embed_documents(texts)  # normalized

    def scores(self, query: str) -> np.ndarray:
        q = self.embedder.embed_query(query)                   # normalized
        return self.doc_embeddings @ q                         # cosine sim


def _minmax(x: np.ndarray) -> np.ndarray:
    lo, hi = x.min(), x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


class HybridRetriever:
    """Weighted score fusion:  S = alpha * S_sparse + (1-alpha) * S_dense.

    Both score vectors are min-max normalized per query before mixing
    (BM25 scores are unbounded; cosine lives in [-1, 1]).
    alpha = 0.3 per arXiv:2407.01219. alpha=1 -> pure sparse,
    alpha=0 -> pure dense, so this one class covers all three modes.
    """

    def __init__(self, documents: List[Document],
                 embedder: HuggingFaceEmbedder, alpha: float):
        self.documents = documents
        self.alpha = alpha
        self.sparse = SparseIndex(documents) if alpha > 0 else None
        self.dense = DenseIndex(documents, embedder) if alpha < 1 else None

    def invoke(self, query: str, k: int = 8,
               allowed_row_id: str = None) -> List[Document]:
        if self.sparse and self.dense:
            s = _minmax(self.sparse.scores(query))
            d = _minmax(self.dense.scores(query))
            fused = self.alpha * s + (1 - self.alpha) * d
        elif self.sparse:
            fused = self.sparse.scores(query)
        else:
            fused = self.dense.scores(query)

        if allowed_row_id is not None:
            # Restrict to chunks belonging to this question's own document(s).
            # Cuad questions are about ONE contract; global retrieval mixes
            # contracts and tanks adherence (cross-contract contamination).
            allowed_idx = [
                i for i, doc in enumerate(self.documents)
                if allowed_row_id in doc.metadata.get("row_ids", "").split(",")
            ]
            if allowed_idx:
                allowed_idx = np.array(allowed_idx)
                order = allowed_idx[np.argsort(fused[allowed_idx])[::-1][:k]]
                return [self.documents[i] for i in order]
            # fallback: no chunks matched (shouldn't happen) -> global top-k

        top_k = np.argsort(fused)[::-1][:k]
        return [self.documents[i] for i in top_k]


class VectorDBAndEmbeddingAlongRetriever:
    """Chunk documents, build the retriever selected in config."""

    def __init__(self, raw_documents: List[Document], config: Dict[str, Any]):
        print("\nData chunking started...")
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=config["chunk_size"],
            chunk_overlap=config["chunk_overlap"],
            length_function=len,
            separators=["\n\n", "\n", ". ", "; ", " ", ""],  # clause-friendly
        )
        self.chunks = splitter.split_documents(raw_documents)
        print(f"Data chunking finished: {len(self.chunks)} chunks.")

        rtype = config["retriever_type"].lower()
        alpha = {"dense": 0.0, "sparse": 1.0}.get(
            rtype, config.get("hybrid_alpha", 0.3))

        embedder = None
        if alpha < 1.0:
            embedder = HuggingFaceEmbedder(config["embedding_model"])

        print(f"Building retriever (type={rtype}, alpha={alpha})...")
        self.retriever = HybridRetriever(self.chunks, embedder, alpha)
        print("Retriever ready.")

    def get_related_docs(self, query: str, k: int,
                         allowed_row_id: str = None) -> List[Document]:
        return self.retriever.invoke(query, k=k, allowed_row_id=allowed_row_id)

**Rerank + repack** — default is now `reverse` (best chunk adjacent to the question).

In [8]:
class RankAndRepackDocuments:
    def __init__(self, config: Dict[str, Any]):
        self.rank_engine = Ranker(model_name=config["ranking_model"],
                                  cache_dir="/tmp")

    def get_documents(self, retrieved_docs: List[Document], query: str,
                      config: Dict[str, Any]) -> List[Document]:
        passages = [
            {"id": idx, "text": doc.page_content, "meta": doc.metadata}
            for idx, doc in enumerate(retrieved_docs)
        ]
        ranked = self.rank_engine.rerank(
            RerankRequest(query=query, passages=passages))

        rerank_k = config.get("rerank_k", 4)
        reranked_docs = [
            Document(page_content=r["text"], metadata=r["meta"])
            for r in ranked[:rerank_k]
        ]

        strategy = config.get("repacking_strategy", "reverse")
        if strategy == "sides":
            repacked = [None] * len(reranked_docs)
            left, right = 0, len(reranked_docs) - 1
            for i, doc in enumerate(reranked_docs):
                if i % 2 == 0:
                    repacked[left] = doc; left += 1
                else:
                    repacked[right] = doc; right -= 1
            return [d for d in repacked if d is not None]
        # "reverse": ascending relevance — best chunk ends up adjacent
        # to the question at the bottom of the prompt (paper's winner)
        return reranked_docs[::-1]

**RAG main pipeline** — legal-tuned generation prompt + RAGBench judge prompt.

In [9]:
class RAGExperimentPipeline:
    def __init__(self, raw_documents: List[Document], config: Dict[str, Any]):
        self.data_store = VectorDBAndEmbeddingAlongRetriever(raw_documents, config)
        self.re_rank_and_repack = RankAndRepackDocuments(config)
        print("Pipeline initialized.")

    def retrieve_docs(self, query: str, config: Dict[str, Any],
                      allowed_row_id: str = None) -> List[Document]:
        related = self.data_store.get_related_docs(
            query, config["top_k"], allowed_row_id=allowed_row_id)
        return self.re_rank_and_repack.get_documents(related, query, config)

    def run_answer_generation(self, query: str, context_docs: List[Document],
                              llm: ChatGroq, config: Dict[str, Any]) -> str:
        context_text = "\n\n".join(
            f"[Document {i}]\n{d.page_content}"
            for i, d in enumerate(context_docs)
        )
        prompt = ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(
                "You are a meticulous legal research assistant. You answer "
                "questions about contracts and legal documents using ONLY "
                "the provided context."
            ),
            HumanMessagePromptTemplate.from_template("""
Answer the question using ONLY the context below. Follow these rules:

1. Ground every statement in the context. Do not add outside legal
   knowledge, statutes, or case law that is not in the context.
2. Be COMPLETE: if several parts of the context are relevant (multiple
   clauses, conditions, exceptions, or parties), include ALL of them.
3. Preserve exact legal terms, defined terms, section numbers, dates,
   and party names as written in the context.
4. Give ONE direct, consolidated answer. Do NOT describe or compare
   documents individually. Never write "Document 0 states" or similar.
5. The answer may be spread across several sentences or clauses — read
   them together and synthesize. If the context contains ANY relevant
   information, ANSWER with what is available, even if partial. State
   what the context establishes. Do NOT refuse merely because the
   answer is incomplete, scattered, or requires connecting clauses.
6. Refuse ONLY if the context contains nothing at all relevant to the
   question. In that rare case, reply exactly:
   "The provided context does not contain enough information to answer
   this question."
7. Be concise: no preamble, no repetition of the question.

Context:
{context}

Question: {question}
Answer:"""),
        ])
        chain = prompt | llm | StrOutputParser()
        return chain.invoke({"context": context_text, "question": query})

    def run_judge_llm(self, query: str, documents: str, generated_response: str,
                      llm: ChatGroq, config: Dict[str, Any]) -> str:
        prompt = ChatPromptTemplate.from_messages([
            SystemMessagePromptTemplate.from_template(
                "You are a strict RAG evaluation judge. Return only valid JSON."),
            HumanMessagePromptTemplate.from_template("""
You are an expert evaluator for Retrieval-Augmented Generation (RAG) systems. Your task is to review a response provided for a given question based on one or more source documents and evaluate alignment, relevance, and hallucination sentence by sentence.

Here are the source documents, split into sentences with unique keys (e.g., '0a.', '0b.'):
{context_documents}

The user question was:
{user_question}

Here is the provided response, split into sentences with unique keys (e.g., 'a.', 'b.'):
{llm_generated_answer}

Evaluate the inputs and output a single, valid JSON object matching this exact schema. Do not include markdown code blocks (```json), backticks, or any conversational text before or after the JSON string. Escape all nested quotes (\\") and newlines (\\n).

{{
  "relevance_explanation": "string",
  "all_relevant_sentence_keys": ["string"],
  "overall_supported_explanation": "string",
  "overall_supported": boolean,
  "sentence_support_information": [
    {{
      "response_sentence_key": "string",
      "explanation": "string",
      "supporting_sentence_keys": ["string"],
      "fully_supported": boolean
    }}
  ],
  "all_utilized_sentence_keys": ["string"]
}}

---
FIELD DEFINITIONS & STRICT LOGIC RULES:

1. "relevance_explanation"
- Content: A step-by-step breakdown explaining which source documents contain useful information for answering the question and exactly how that information is helpful.

2. "all_relevant_sentence_keys"
- Content: An array of all document sentence keys relevant to the question.
- Rule: Include every sentence useful to the question, even if it was completely omitted from the provided response, or if only a portion of it is useful. Base this judgment strictly on the source documents and the question; ignore the provided response entirely. Omit sentences that could be deleted without impacting a human's ability to answer the question.

3. "overall_supported_explanation"
- Content: A step-by-step evaluation of why the response as a whole is or is not supported by the documents.
- Constraint: You MUST evaluate each response claim separately, one by one, in isolation first. Do not make any summary remarks about the response as a whole until all isolated claims have been completely assessed.

4. "overall_supported"
- Content: Boolean (true/false) indicating if the entire response is supported. This must logically match the final conclusion drawn in "overall_supported_explanation".

5. "sentence_support_information"
- Content: A list containing exactly one object per sentence in the provided response.
  * "response_sentence_key": Matches the key of the sentence from the provided response.
  * "explanation": A detailed string explaining why this specific sentence is or is not supported by the source text.
  * "supporting_sentence_keys": Array of keys from the source documents that support this specific response sentence.
    - If the sentence is NOT supported, this array MUST be empty.
    - If the sentence IS supported, provide the source keys.
    - Special Case Exceptions: If a sentence is supported but has no specific source key, populate this field with one of these literal string identifiers instead:
      - "supported_without_sentence": If the response sentence expresses an inability to answer due to missing context information, or is supported generally by the collective text.
      - "general": For transition sentences, summaries of previous sentences, or outlines of the answering steps.
      - "well_known_fact": If the sentence states a universally known fact (e.g., a math formula).
      - "numerical_reasoning": If the sentence executes basic math logic (e.g., addition, multiplication).
  * "fully_supported": Boolean (true/false).
    - If "supporting_sentence_keys" is empty, this MUST be false.
    - Set to true only if every single claim within the response sentence is perfectly covered by the keys in "supporting_sentence_keys". Set to false if it is only partially or incompletely supported.

6. "all_utilized_sentence_keys"
- Content: An array of all source document sentence keys that were actively used to construct the response.
- Rule: Include keys that directly supported the answer or were implicitly used to build it (even if the source sentence was not used in its entirety). Omit source keys that were completely ignored or had no bearing on the final answer.
"""),
        ])
        chain = prompt | llm | StrOutputParser()
        return chain.invoke({
            "context_documents": documents,
            "user_question": query,
            "llm_generated_answer": generated_response,
        })

**Load dataset & build pipeline** (chunk + index happens once here — takes a few minutes for embeddings)

In [10]:
dataset = load_dataset("rungalileo/ragbench", config["data_set"])
test_data = dataset[config["data_type"]]
df = test_data.to_pandas()
print(f"Loaded {config['data_set']} / {config['data_type']}: {len(test_data)} rows")

raw_documents = convert_dataframe_to_langchain_docs(df)
rag_pipeline = RAGExperimentPipeline(raw_documents, config)

README.md:   0%|          | 0.00/24.7k [00:00<?, ?B/s]

cuad/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 56.4MB            

cuad/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

cuad/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 15.7MB            

cuad/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

Building retriever (type=hybrid, alpha=0.3)...


Batches:   0%|          | 0/103 [00:00<?, ?it/s]

Retriever ready.


ms-marco-MiniLM-L-12-v2.zip: 100%|██████████| 21.6M/21.6M [00:00<00:00, 30.0MiB/s]


Pipeline initialized.


**Restore checkpoint from Drive** (if resuming a previous run)

In [11]:
for src_f, dst_f in [(CHECKPOINT_DRIVE_FILE, CHECKPOINT_FILE),
                     (FINAL_DRIVE_FILE, FINAL_FILE)]:
    try:
        shutil.copy(src_f, dst_f)
        print("Restored from Drive:", dst_f)
    except Exception:
        print("Not found on Drive (fresh run):", dst_f)

if os.path.exists(CHECKPOINT_FILE):
    existing = pd.read_csv(CHECKPOINT_FILE)
    completed_ids = (
        set(existing[existing["error"].isna()]["row_id"].astype(int))
        if "row_id" in existing.columns else set()
    )
    results = existing.to_dict("records")
    print(f"Checkpoint found. {len(completed_ids)} rows already done.")
else:
    completed_ids, results = set(), []
    print("No checkpoint. Starting fresh.")


def is_checkpoint_required(last_save_time, duration):
    now = time.time()
    if now - last_save_time >= duration:
        return True, now
    return False, last_save_time


def move_records_to_drive(total_rows: int):
    results_df = pd.read_csv(CHECKPOINT_FILE)
    successful = results_df[results_df["error"].isna()]
    if successful.shape[0] == total_rows:
        successful.to_csv(FINAL_FILE, index=False)
        shutil.copy(FINAL_FILE, FINAL_DRIVE_FILE)
        print("Full run complete. Final file copied to Drive:", FINAL_FILE)
    else:
        shutil.copy(CHECKPOINT_FILE, CHECKPOINT_DRIVE_FILE)
        print("Checkpoint copied to Drive. Resume later.")

Restored from Drive: ragbench_cuad_test_1000_200_hybrid_12_reverse_v3_softprompt_checkpoint.csv
Not found on Drive (fresh run): ragbench_cuad_test_1000_200_hybrid_12_reverse_v3_softprompt_final.csv
Checkpoint found. 420 rows already done.


**RAG execution**

Tip: for a smoke test before the full run, temporarily change
`range(len(test_data))` to `range(50)` and confirm metrics look sane.

In [13]:
answer_llm = ChatGroq(model=config["generator_model"], temperature=0)
judge_llm = ChatGroq(model=config["judge_model"], temperature=0)

save_interval = 300
last_save_time = time.time()

def is_daily_limit_error(error):
    return "tokens per day" in str(error).lower() or "tpd" in str(error).lower()

for i in tqdm(range(len(test_data)), desc="Evaluating"):
    if i in completed_ids:
        continue

    row = test_data[i]
    question = row["question"]

    try:
        # 1. Retrieve -> rerank -> repack (restricted to this row's own documents)
        retrieved_docs = rag_pipeline.retrieve_docs(
            question, config, allowed_row_id=str(row.get("id", "N/A")))

        # 2. Generate answer
        generated_response = rag_pipeline.run_answer_generation(
            question, retrieved_docs, answer_llm, config)

        # 3. Key context + response sentences
        keyed_context, context_for_judge = key_context_sentences(retrieved_docs)
        keyed_response, response_for_judge = key_response_sentences(generated_response)

        # 4. Judge
        judge_response = rag_pipeline.run_judge_llm(
            question, context_for_judge, response_for_judge, judge_llm, config)
        judge_stats = parse_json_safely(judge_response)
        if judge_stats is None:
            raise ValueError("Judge output could not be parsed as JSON")

        # 5. Metrics (keyed_context passed in full so keys can be validated)
        total_context_sentence_count = len(keyed_context)
        pred = compute_trace_metrics(judge_stats, keyed_context)

        # 6. Store
        results.append({
            "row_id": i,
            "question": question,
            "generated_response": generated_response,
            "retrieved_doc_ids": json.dumps(
                [doc.metadata["id"] for doc in retrieved_docs]),
            "judge_output": json.dumps(judge_stats),
            "total_context_sentence_count": total_context_sentence_count,
            "pred_relevance_score": pred["pred_relevance_score"],
            "pred_utilization_score": pred["pred_utilization_score"],
            "pred_completeness_score": pred["pred_completeness_score"],
            "pred_adherence_score": pred["pred_adherence_score"],
            "pred_adherence_continuous": pred["pred_adherence_continuous"],
            "gold_relevance_score": row.get("relevance_score", None),
            "gold_utilization_score": row.get("utilization_score", None),
            "gold_completeness_score": row.get("completeness_score", None),
            "gold_adherence_score": row.get("adherence_score", None),
            "error": None,
        })
        completed_ids.add(i)
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)

        need_save, last_save_time = is_checkpoint_required(last_save_time, save_interval)
        if need_save:
            move_records_to_drive(len(test_data))

        time.sleep(1)  # be gentle to Groq free tier

    except Exception as e:
        if is_rate_limit_error(e):
            if not is_daily_limit_error(e):
                # Per-minute limit: resolves in seconds — retry same row, same key
                print(f"\nTPM limit on row {i}, sleeping 20s and retrying...")
                time.sleep(20)
                continue
            print("\nDaily token limit hit:", e)
            if len(secret_keys) > secret_keys_index + 1:
                secret_keys_index += 1
                os.environ["GROQ_API_KEY"] = secret_keys[secret_keys_index]
                answer_llm = ChatGroq(model=config["generator_model"], temperature=0)
                judge_llm = ChatGroq(model=config["judge_model"], temperature=0)
                print("Switched to next Groq key. Continuing.")
                continue
            print("No keys left. Saving checkpoint and stopping.")
            pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
            move_records_to_drive(len(test_data))
            print("Next row to resume from:", i)
            break

        print(f"\nError on row {i}: {e}")
        results.append({
            "row_id": i, "question": question,
            "generated_response": None, "retrieved_doc_ids": None,
            "judge_output": None, "total_context_sentence_count": None,
            "pred_relevance_score": None, "pred_utilization_score": None,
            "pred_completeness_score": None, "pred_adherence_score": None,
            "pred_adherence_continuous": None,
            "gold_relevance_score": row.get("relevance_score", None),
            "gold_utilization_score": row.get("utilization_score", None),
            "gold_completeness_score": row.get("completeness_score", None),
            "gold_adherence_score": row.get("adherence_score", None),
            "error": str(e),
        })
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        continue

move_records_to_drive(len(test_data))
print("\nRun finished or stopped safely. Checkpoint:", CHECKPOINT_FILE)

Evaluating:  95%|█████████▌| 485/510 [01:00<00:08,  3.07it/s]


Daily token limit hit: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kwr9ys0mfjwse171kpvqv7qa` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 97635, Requested 3166. Please try again in 11m32.064s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Switched to next Groq key. Continuing.


Evaluating: 100%|██████████| 510/510 [04:44<00:00,  1.79it/s]

Checkpoint copied to Drive. Resume later.

Run finished or stopped safely. Checkpoint: ragbench_cuad_test_1000_200_hybrid_12_reverse_v3_softprompt_checkpoint.csv


In [14]:
chk = pd.read_csv(CHECKPOINT_FILE)
done = chk[chk["error"].isna()]
print(f"Successful rows: {len(done)} / {len(chk)} saved")
print(f"Full test set size: {len(test_data)}")
print(chk[["row_id", "pred_relevance_score", "pred_adherence_score", "error"]].tail(10))

Successful rows: 509 / 509 saved
Full test set size: 510
     row_id  pred_relevance_score  pred_adherence_score  error
499     500              0.400000                  True    NaN
500     501              0.272727                 False    NaN
501     502              0.333333                 False    NaN
502     503              0.238095                  True    NaN
503     504              0.272727                 False    NaN
504     505              0.346154                 False    NaN
505     506              0.000000                  True    NaN
506     507              0.062500                  True    NaN
507     508              0.312500                  True    NaN
508     509              0.000000                  True    NaN


**Model performance summary** — same JSON schema as v1, drops straight into the team comparison table.

In [16]:
results_df = pd.read_csv(CHECKPOINT_FILE)
results_df["pred_completeness_score"] = results_df["pred_completeness_score"].fillna(0)

metrics_summary = {k: config[k] for k in [
    "data_set", "data_type", "chunk_size", "chunk_overlap",
    "embedding_model", "retriever_type", "top_k", "ranking_model",
    "rerank_k", "repacking_strategy", "generator_model", "judge_model"]}

metrics_summary["mse_relevance"] = mean_squared_error(
    results_df["gold_relevance_score"], results_df["pred_relevance_score"])
metrics_summary["rmse_relevance"] = root_mean_squared_error(
    results_df["gold_relevance_score"], results_df["pred_relevance_score"])
metrics_summary["mse_utilization"] = mean_squared_error(
    results_df["gold_utilization_score"], results_df["pred_utilization_score"])
metrics_summary["rmse_utilization"] = root_mean_squared_error(
    results_df["gold_utilization_score"], results_df["pred_utilization_score"])
metrics_summary["mse_completeness"] = mean_squared_error(
    results_df["gold_completeness_score"], results_df["pred_completeness_score"])
metrics_summary["rmse_completeness"] = root_mean_squared_error(
    results_df["gold_completeness_score"], results_df["pred_completeness_score"])
metrics_summary["accuracy_adherence"] = accuracy_score(
    results_df["gold_adherence_score"], results_df["pred_adherence_score"])
metrics_summary["f1_score_adherence"] = f1_score(
    results_df["gold_adherence_score"], results_df["pred_adherence_score"])

with open(SUMMARY_FILE, "w") as f:
    json.dump(metrics_summary, f, indent=2)

shutil.copy(SUMMARY_FILE, SUMMARY_DRIVE_FILE)
print("Summary saved and copied to Drive.")
print(json.dumps(metrics_summary, indent=2))

Summary saved and copied to Drive.
{
  "data_set": "cuad",
  "data_type": "test",
  "chunk_size": 1000,
  "chunk_overlap": 200,
  "embedding_model": "BAAI/bge-base-en-v1.5",
  "retriever_type": "hybrid",
  "top_k": 12,
  "ranking_model": "ms-marco-MiniLM-L-12-v2",
  "rerank_k": 4,
  "repacking_strategy": "reverse",
  "generator_model": "llama-3.1-8b-instant",
  "judge_model": "llama-3.3-70b-versatile",
  "mse_relevance": 0.1168338306201472,
  "rmse_relevance": 0.3418096409116443,
  "mse_utilization": 0.0267956758842764,
  "rmse_utilization": 0.16369384803429968,
  "mse_completeness": 0.3703447582484496,
  "rmse_completeness": 0.6085595765810029,
  "accuracy_adherence": 0.630648330058939,
  "f1_score_adherence": 0.7602040816326531
}
